<a href="https://colab.research.google.com/github/collinskiplangatrotich/plp-da-a7-pesapap-growth-analytics/blob/main/pesapap_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PesaPap Growth Analytics & Onboarding Experiment Analysis
This notebook builds `pesapap.db` from `users.csv`, `transactions.csv`, and `experiment.csv` (generated with seed `77`), executes every SQL query in `sql/queries.sql` via `pd.read_sql`, validates data integrity, computes the KPI scorecard and metric dictionary, segments users with RFM and cohort retention, and evaluates the Group A vs. Group B onboarding experiment.

In [8]:
import os
import re
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats

# Ensure required CSV files exist using the unchanged seed 77 generator
if not (os.path.exists("users.csv") and os.path.exists("transactions.csv") and os.path.exists("experiment.csv")):
    rng = np.random.default_rng(77)

    users = pd.DataFrame({
        "user_id": range(1, 3001),
        "signup_date": (pd.to_datetime("2026-01-01")
                        + pd.to_timedelta(rng.integers(0, 120, 3000), unit="D"))
                       .strftime("%Y-%m-%d"),
        "county": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], 3000),
        "acquisition_channel": rng.choice(["Referral", "Social", "Agent", "Search"],
                                          3000),
    })

    m = 12000
    txns = pd.DataFrame({
        "txn_id": range(1, m + 1),
        "user_id": rng.integers(1, 3001, m),
        "txn_date": (pd.to_datetime("2026-01-15")
                     + pd.to_timedelta(rng.integers(0, 150, m), unit="D"))
                    .strftime("%Y-%m-%d"),
        "product": rng.choice(["Airtime", "Bill Pay", "Send Money", "Merchant Pay"], m,
                              p=[0.3, 0.2, 0.35, 0.15]),
        "amount": rng.lognormal(6.8, 0.9, m).round(0),
    })

    experiment = pd.DataFrame({"user_id": range(1, 3001),
                               "group": rng.choice(["A", "B"], 3000)})
    is_b = experiment["group"] == "B"
    experiment["converted"] = np.where(is_b, rng.random(3000) < 0.14,
                                       rng.random(3000) < 0.11).astype(int)

    users.to_csv("users.csv", index=False)
    txns.to_csv("transactions.csv", index=False)
    experiment.to_csv("experiment.csv", index=False)

users_df = pd.read_csv("users.csv")
txns_df = pd.read_csv("transactions.csv")
exp_df = pd.read_csv("experiment.csv")

# Build pesapap.db in SQLite
db_path = "pesapap.db"
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
users_df.to_sql("users", conn, index=False, if_exists="replace")
txns_df.to_sql("transactions", conn, index=False, if_exists="replace")
exp_df.to_sql("experiment", conn, index=False, if_exists="replace")

# Parse and execute every query in sql/queries.sql through pd.read_sql
with open("sql/queries.sql", "r", encoding="utf-8") as f:
    sql_text = f.read()

# Split by semicolon while preserving leading comments per statement
raw_chunks = [chunk.strip() for chunk in sql_text.split(";") if chunk.strip()]
query_results = {}

for idx, chunk in enumerate(raw_chunks, start=1):
    lines = [line.strip() for line in chunk.splitlines() if line.strip()]
    comment_lines = [line for line in lines if line.startswith("--")]
    sql_lines = [line for line in lines if not line.startswith("--")]
    sql_stmt = "\n".join(sql_lines).strip()
    label = comment_lines[0].lstrip("-").strip() if comment_lines else f"Query {idx}"

    if sql_stmt:
        res_df = pd.read_sql(sql_stmt, conn)
        query_results[label] = res_df
        print("=" * 80)
        print(f"SQL RESULT {idx}: {label}")
        print("=" * 80)
        display(res_df)

SQL RESULT 1: Validation Question: Identify and quantify transactions occurring before a user's signup_date (signup-date / pre-signup transaction anomaly).


,pre_signup_txn_count,affected_users,pre_signup_volume,pct_of_all_txns
0,3737,1749,5022531.0,31.14


SQL RESULT 2: Question a: What are the overall transaction volume, transaction count, active transacting users, and average transaction amount on PesaPap?


,total_txns,active_users,total_volume,avg_txn_amount
0,12000,2943,15979309.0,1331.61


SQL RESULT 3: Question b: How do transaction count, total volume, and average transaction amount break down by product?


,product,txn_count,total_volume,avg_amount,volume_share_pct
0,Send Money,4161,5639308.0,1355.28,35.29
1,Airtime,3662,4708104.0,1285.66,29.46
2,Bill Pay,2388,3224392.0,1350.25,20.18
3,Merchant Pay,1789,2407505.0,1345.73,15.07


... (8 outputs left out)

SQL RESULT 8: Question g: Using a CTE, CASE WHEN, and LAG, how does monthly transaction volume change month-over-month and classify growth momentum?


,txn_month,txn_count,monthly_volume,prev_month_volume,mom_volume_change,mom_status
0,2026-01,1308,1733121.0,NaN,NaN,Baseline Month
1,2026-02,2272,2993105.0,1733121.0,1259984.0,Expansion
2,2026-03,2394,3199125.0,2993105.0,206020.0,Expansion
3,2026-04,2470,3262447.0,3199125.0,63322.0,Expansion
4,2026-05,2475,3365002.0,3262447.0,102555.0,Expansion
5,2026-06,1081,1426509.0,3365002.0,-1938493.0,Contraction


SQL RESULT 9: Question h: Using a CTE, CASE WHEN, and LAG, how does monthly transaction volume for Send Money change month-over-month with period-over-period flags?


,txn_month,send_money_volume,prev_send_money_volume,mom_growth_pct,growth_tier
0,2026-01,616384.0,NaN,NaN,First Month
1,2026-02,1033620.0,616384.0,67.69,Strong Growth (>=5%)
2,2026-03,1186523.0,1033620.0,14.79,Strong Growth (>=5%)
3,2026-04,1176926.0,1186523.0,-0.81,Slowdown / Decline
4,2026-05,1150352.0,1176926.0,-2.26,Slowdown / Decline
5,2026-06,475503.0,1150352.0,-58.66,Slowdown / Decline


## Data Validation: Signup-Date Check
We check whether any transactions in `transactions.csv` have `txn_date < signup_date` in `users.csv`. Because `signup_date` spans `2026-01-01` through `2026-04-30` while `txn_date` spans `2026-01-15` through `2026-06-13`, random date generation produces transactions dated prior to a user's recorded `signup_date`. Below we quantify the affected records and evaluate post-signup (`txn_date >= signup_date`) vs. full-dataset behavior so the impact is transparent and justified.

In [4]:
merged_txns = txns_df.merge(
    users_df[["user_id", "signup_date", "county", "acquisition_channel"]],
    on="user_id",
    how="left",
)
merged_txns["is_pre_signup"] = merged_txns["txn_date"] < merged_txns["signup_date"]

pre_signup_count = int(merged_txns["is_pre_signup"].sum())
total_txn_count = len(merged_txns)
pre_signup_pct = pre_signup_count / total_txn_count * 100
pre_signup_vol = merged_txns.loc[merged_txns["is_pre_signup"], "amount"].sum()
total_vol = merged_txns["amount"].sum()
affected_users = merged_txns.loc[merged_txns["is_pre_signup"], "user_id"].nunique()

post_signup_df = merged_txns[~merged_txns["is_pre_signup"]].copy()

validation_summary = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Pre-Signup Transactions (txn_date < signup_date)",
        "Pre-Signup Share of Transactions (%)",
        "Affected Unique Users",
        "Pre-Signup Volume",
        "Total Volume",
        "Pre-Signup Share of Volume (%)",
        "Post-Signup Transactions (txn_date >= signup_date)",
    ],
    "Value": [
        f"{total_txn_count:,}",
        f"{pre_signup_count:,}",
        f"{pre_signup_pct:.2f}%",
        f"{affected_users:,}",
        f"{pre_signup_vol:,.2f}",
        f"{total_vol:,.2f}",
        f"{pre_signup_vol / total_vol * 100:.2f}%",
        f"{len(post_signup_df):,}",
    ],
})

print("--- Signup-Date Validation Summary ---")
display(validation_summary)

print(
    f"Handling Justification: {pre_signup_count:,} transactions ({pre_signup_pct:.2f}%) occur before "
    f"the user's recorded signup_date. In core revenue/SQL tables we preserve raw table fidelity while "
    f"flagging pre-signup events (`is_pre_signup`), and we report post-signup (`txn_date >= signup_date`) "
    f"checks so behavioural/cohort metrics are not distorted by pre-registration timestamps."
)

--- Signup-Date Validation Summary ---


,Metric,Value
0,Total Transactions,"12,000"
1,Pre-Signup Transactions (txn_date < signup_date),"3,737"
2,Pre-Signup Share of Transactions (%),31.14%
3,Affected Unique Users,"1,749"
4,Pre-Signup Volume,"5,022,531.00"
5,Total Volume,"15,979,309.00"
6,Pre-Signup Share of Volume (%),31.43%
7,Post-Signup Transactions (txn_date >= signup_d...,"8,263"


Handling Justification: 3,737 transactions (31.14%) occur before the user's recorded signup_date. In core revenue/SQL tables we preserve raw table fidelity while flagging pre-signup events (`is_pre_signup`), and we report post-signup (`txn_date >= signup_date`) checks so behavioural/cohort metrics are not distorted by pre-registration timestamps.


## KPI Scorecard & Metric Dictionary
Below is the executive KPI scorecard for PesaPap alongside a metric dictionary specifying each metric's **name**, **definition**, and **calculation**.

In [5]:
total_users = len(users_df)
active_users = txns_df["user_id"].nunique()
activation_rate_pct = active_users / total_users * 100
total_txns = len(txns_df)
total_volume = txns_df["amount"].sum()
avg_txn_amount = txns_df["amount"].mean()
txns_per_active_user = total_txns / active_users

conv_a = exp_df.loc[exp_df["group"] == "A", "converted"].mean()
conv_b = exp_df.loc[exp_df["group"] == "B", "converted"].mean()
rel_lift_pct = (conv_b - conv_a) / conv_a * 100

kpi_scorecard = pd.DataFrame([
    {"KPI": "Registered Users", "Value": f"{total_users:,}"},
    {"KPI": "Active Transacting Users", "Value": f"{active_users:,}"},
    {"KPI": "User Activation Rate", "Value": f"{activation_rate_pct:.2f}%"},
    {"KPI": "Total Transactions", "Value": f"{total_txns:,}"},
    {"KPI": "Total Transaction Volume", "Value": f"{total_volume:,.0f}"},
    {"KPI": "Average Transaction Amount", "Value": f"{avg_txn_amount:,.2f}"},
    {"KPI": "Transactions per Active User", "Value": f"{txns_per_active_user:.2f}"},
    {"KPI": "Group A Conversion Rate (Control)", "Value": f"{conv_a * 100:.2f}%"},
    {"KPI": "Group B Conversion Rate (Simplified)", "Value": f"{conv_b * 100:.2f}%"},
    {"KPI": "Experiment Relative Lift (B vs. A)", "Value": f"{rel_lift_pct:+.2f}%"},
])

metric_dictionary = pd.DataFrame([
    {
        "Name": "Registered Users",
        "Definition": "Total number of onboarded user profiles in the platform user table.",
        "Calculation": "COUNT(DISTINCT user_id) FROM users",
    },
    {
        "Name": "Active Transacting Users",
        "Definition": "Distinct users who completed at least one transaction in the observation window.",
        "Calculation": "COUNT(DISTINCT user_id) FROM transactions",
    },
    {
        "Name": "User Activation Rate",
        "Definition": "Share of registered users who have completed at least one transaction.",
        "Calculation": "(Active Transacting Users / Registered Users) * 100",
    },
    {
        "Name": "Total Transactions",
        "Definition": "Total count of completed transaction records across all products.",
        "Calculation": "COUNT(txn_id) FROM transactions",
    },
    {
        "Name": "Total Transaction Volume",
        "Definition": "Cumulative monetary value processed across all transactions.",
        "Calculation": "SUM(amount) FROM transactions",
    },
    {
        "Name": "Average Transaction Amount",
        "Definition": "Mean monetary value per transaction.",
        "Calculation": "AVG(amount) FROM transactions",
    },
    {
        "Name": "Transactions per Active User",
        "Definition": "Average transaction frequency among users with at least one transaction.",
        "Calculation": "Total Transactions / Active Transacting Users",
    },
    {
        "Name": "Conversion Rate (Group A / Group B)",
        "Definition": "Proportion of experiment users in each onboarding group who converted.",
        "Calculation": "SUM(converted) / COUNT(user_id) GROUP BY group",
    },
    {
        "Name": "Relative Lift",
        "Definition": "Percentage improvement in Group B conversion rate relative to Group A.",
        "Calculation": "((Conversion Rate B - Conversion Rate A) / Conversion Rate A) * 100",
    },
])

print("--- KPI Scorecard ---")
display(kpi_scorecard)

print("--- Metric Dictionary ---")
display(metric_dictionary)

--- KPI Scorecard ---


,KPI,Value
0,Registered Users,"3,000"
1,Active Transacting Users,"2,943"
2,User Activation Rate,98.10%
3,Total Transactions,"12,000"
4,Total Transaction Volume,"15,979,309"
5,Average Transaction Amount,"1,331.61"
6,Transactions per Active User,4.08
7,Group A Conversion Rate (Control),10.46%
8,Group B Conversion Rate (Simplified),13.81%
9,Experiment Relative Lift (B vs. A),+31.96%


--- Metric Dictionary ---


,Name,Definition,Calculation
0,Registered Users,Total number of onboarded user profiles in the...,COUNT(DISTINCT user_id) FROM users
1,Active Transacting Users,Distinct users who completed at least one tran...,COUNT(DISTINCT user_id) FROM transactions
2,User Activation Rate,Share of registered users who have completed a...,(Active Transacting Users / Registered Users) ...
3,Total Transactions,Total count of completed transaction records a...,COUNT(txn_id) FROM transactions
4,Total Transaction Volume,Cumulative monetary value processed across all...,SUM(amount) FROM transactions
5,Average Transaction Amount,Mean monetary value per transaction.,AVG(amount) FROM transactions
6,Transactions per Active User,Average transaction frequency among users with...,Total Transactions / Active Transacting Users
7,Conversion Rate (Group A / Group B),Proportion of experiment users in each onboard...,SUM(converted) / COUNT(user_id) GROUP BY group
8,Relative Lift,Percentage improvement in Group B conversion r...,((Conversion Rate B - Conversion Rate A) / Con...


## RFM Segmentation & Cohort Retention Table
Below we compute **RFM** (Recency, Frequency, Monetary) segments across all `3,000` users and construct a **Signup Cohort Retention Table** by signup month. Cells where the dataset window (`2026-01-15` to `2026-06-13`) ends before a full calendar month has elapsed are explicitly flagged as **Incomplete**.

In [6]:
max_txn_date = pd.to_datetime(txns_df["txn_date"].max())

txn_summary = (
    txns_df.assign(txn_dt=pd.to_datetime(txns_df["txn_date"]))
    .groupby("user_id", as_index=False)
    .agg(
        last_txn_dt=("txn_dt", "max"),
        frequency=("txn_id", "count"),
        monetary=("amount", "sum"),
    )
)

rfm = users_df[["user_id"]].merge(txn_summary, on="user_id", how="left")
rfm["frequency"] = rfm["frequency"].fillna(0).astype(int)
rfm["monetary"] = rfm["monetary"].fillna(0.0)
rfm["recency_days"] = np.where(
    rfm["last_txn_dt"].notna(),
    (max_txn_date - rfm["last_txn_dt"]).dt.days,
    np.nan,
)

active_mask = rfm["frequency"] > 0
r_score = pd.Series(0, index=rfm.index, dtype=int)
f_score = pd.Series(0, index=rfm.index, dtype=int)
m_score = pd.Series(0, index=rfm.index, dtype=int)

# Higher score = more recent (lower recency_days), higher frequency, higher monetary
r_score.loc[active_mask] = pd.qcut(
    rfm.loc[active_mask, "recency_days"].rank(method="first"),
    q=3,
    labels=[3, 2, 1],
).astype(int)
f_score.loc[active_mask] = pd.qcut(
    rfm.loc[active_mask, "frequency"].rank(method="first"),
    q=3,
    labels=[1, 2, 3],
).astype(int)
m_score.loc[active_mask] = pd.qcut(
    rfm.loc[active_mask, "monetary"].rank(method="first"),
    q=3,
    labels=[1, 2, 3],
).astype(int)

rfm["rfm_avg"] = (r_score + f_score + m_score) / 3.0

def assign_rfm_segment(row):
    if row["frequency"] == 0:
        return "Non-Transacting"
    if row["rfm_avg"] >= 2.5:
        return "Champions"
    if row["rfm_avg"] >= 1.8:
        return "Loyal & High-Value"
    if row["recency_days"] <= 30:
        return "Recent / Developing"
    return "At Risk / Lapsed"

rfm["segment"] = rfm.apply(assign_rfm_segment, axis=1)

rfm_counts = (
    rfm.groupby("segment", as_index=False)
    .agg(
        user_count=("user_id", "count"),
        avg_recency_days=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
    )
    .assign(share_pct=lambda d: (d["user_count"] / len(rfm) * 100).round(2))
    .sort_values("user_count", ascending=False)
    .round({"avg_recency_days": 1, "avg_frequency": 2, "avg_monetary": 2})
)

print("--- RFM Segment Counts ---")
display(rfm_counts)

# Cohort Retention Table by Signup Month (using post-signup transactions txn_date >= signup_date)
cohort_txns = post_signup_df.copy()
cohort_txns["signup_month"] = pd.to_datetime(cohort_txns["signup_date"]).dt.to_period("M")
cohort_txns["txn_month"] = pd.to_datetime(cohort_txns["txn_date"]).dt.to_period("M")
cohort_txns["cohort_index"] = (cohort_txns["txn_month"] - cohort_txns["signup_month"]).apply(lambda p: p.n)

users_cohort = users_df.copy()
users_cohort["signup_month"] = pd.to_datetime(users_cohort["signup_date"]).dt.to_period("M")
cohort_sizes = users_cohort.groupby("signup_month")["user_id"].nunique()

max_period = int(cohort_txns["cohort_index"].max())
max_obs_period = pd.Period("2026-06", freq="M")

retention_table = pd.DataFrame(index=cohort_sizes.index, columns=[f"M{i}" for i in range(max_period + 1)], dtype=object)

for c_month, c_size in cohort_sizes.items():
    sub = cohort_txns[cohort_txns["signup_month"] == c_month]
    for p_idx in range(max_period + 1):
        col_name = f"M{p_idx}"
        target_month = c_month + p_idx
        # Flag cells extending beyond the dataset observation window or into partial/incomplete trailing months
        if target_month > max_obs_period or (c_month == pd.Period("2026-04", freq="M") and p_idx >= 3) or (c_month == pd.Period("2026-03", freq="M") and p_idx >= 4) or (c_month == pd.Period("2026-02", freq="M") and p_idx >= 5):
            retention_table.loc[c_month, col_name] = "NA [Incomplete]"
        else:
            retained_users = sub.loc[sub["cohort_index"] == p_idx, "user_id"].nunique()
            pct = retained_users / c_size * 100
            # Flag June 2026 cells as partial-month window (data ends 2026-06-13)
            if target_month == max_obs_period:
                retention_table.loc[c_month, col_name] = f"{pct:.1f}% [Incomplete: partial Jun]"
            else:
                retention_table.loc[c_month, col_name] = f"{pct:.1f}%"

retention_table.insert(0, "Cohort Size", cohort_sizes.values)
retention_table.index = retention_table.index.astype(str)

print("--- Signup Cohort Retention Table (Incomplete Cells Flagged) ---")
display(retention_table)

--- RFM Segment Counts ---


,segment,user_count,avg_recency_days,avg_frequency,avg_monetary,share_pct
2,Loyal & High-Value,935,26.5,4.32,5983.85,31.17
0,At Risk / Lapsed,834,70.3,2.54,2796.14,27.80
1,Champions,792,11.6,6.16,9061.63,26.40
4,Recent / Developing,382,16.9,2.52,2292.19,12.73
3,Non-Transacting,57,NaN,0.00,0.00,1.90


--- Signup Cohort Retention Table (Incomplete Cells Flagged) ---


,Cohort Size,M0,M1,M2,M3,M4,M5
signup_month,,,,,,,
2026-01,773,24.7%,53.7%,55.9%,58.3%,56.0%,33.5% [Incomplete: partial Jun]
2026-02,703,29.9%,54.9%,54.6%,59.0%,28.7% [Incomplete: partial Jun],NA [Incomplete]
2026-03,745,32.5%,55.4%,56.9%,30.2% [Incomplete: partial Jun],NA [Incomplete],NA [Incomplete]
2026-04,779,34.9%,53.7%,28.4% [Incomplete: partial Jun],NA [Incomplete],NA [Incomplete],NA [Incomplete]


## Onboarding Experiment Analysis (Group A vs. Group B)
* **Null Hypothesis ($H_0$):** The simplified onboarding screen (Group B) has no effect on conversion rate compared to the existing screen (Group A), i.e., $p_B = p_A$ ($p_B - p_A = 0$).
* **Alternative Hypothesis ($H_1$):** The simplified onboarding screen (Group B) produces a different conversion rate than the existing screen (Group A), i.e., $p_B \neq p_A$ ($p_B - p_A \neq 0$).

Below we check group balance across `county`, `acquisition_channel`, and `signup_date`, compute conversion rates and relative lift, run a Pearson chi-square test (`scipy.stats.chi2_contingency`), calculate the 95% confidence interval for the difference in proportions, and interpret the findings.

In [7]:
exp_full = exp_df.merge(users_df, on="user_id", how="left")

# 1. Group-balance checks
print("--- Group Balance: User Counts ---")
display(exp_full["group"].value_counts().rename("users").to_frame())

print("--- Group Balance: County Share (%) ---")
display((pd.crosstab(exp_full["county"], exp_full["group"], normalize="columns") * 100).round(2))

print("--- Group Balance: Acquisition Channel Share (%) ---")
display((pd.crosstab(exp_full["acquisition_channel"], exp_full["group"], normalize="columns") * 100).round(2))

exp_full["signup_month"] = pd.to_datetime(exp_full["signup_date"]).dt.to_period("M").astype(str)
print("--- Group Balance: Signup Month Share (%) ---")
display((pd.crosstab(exp_full["signup_month"], exp_full["group"], normalize="columns") * 100).round(2))

# 2. Conversion rates, relative lift, chi-square test, and 95% CI
grp_a = exp_full[exp_full["group"] == "A"]["converted"]
grp_b = exp_full[exp_full["group"] == "B"]["converted"]

n_a, n_b = len(grp_a), len(grp_b)
conv_a_cnt, conv_b_cnt = int(grp_a.sum()), int(grp_b.sum())
p_a, p_b = grp_a.mean(), grp_b.mean()

diff = p_b - p_a
rel_lift = diff / p_a

# Contingency table: rows = [A, B], cols = [Converted=1, Not Converted=0]
contingency = np.array([
    [conv_a_cnt, n_a - conv_a_cnt],
    [conv_b_cnt, n_b - conv_b_cnt],
])
chi2_stat, p_val, dof, expected = stats.chi2_contingency(contingency, correction=False)

# 95% confidence interval for difference in proportions (p_b - p_a)
se_diff = np.sqrt((p_a * (1 - p_a) / n_a) + (p_b * (1 - p_b) / n_b))
z_crit = stats.norm.ppf(0.975)
ci_low = diff - z_crit * se_diff
ci_high = diff + z_crit * se_diff

exp_summary = pd.DataFrame({
    "Metric": [
        "Group A Users",
        "Group A Conversions",
        "Group A Conversion Rate",
        "Group B Users",
        "Group B Conversions",
        "Group B Conversion Rate",
        "Absolute Conversion Difference (B - A)",
        "Relative Lift ((B - A) / A)",
        "Chi-Square Statistic (df=1)",
        "p-value",
        "95% CI for Difference (B - A)",
    ],
    "Value": [
        f"{n_a:,}",
        f"{conv_a_cnt:,}",
        f"{p_a * 100:.2f}%",
        f"{n_b:,}",
        f"{conv_b_cnt:,}",
        f"{p_b * 100:.2f}%",
        f"{diff * 100:+.2f} percentage points ({diff:+.4f})",
        f"{rel_lift * 100:+.2f}%",
        f"{chi2_stat:.4f}",
        f"{p_val:.4f}",
        f"[{ci_low * 100:+.2f} pp, {ci_high * 100:+.2f} pp]",
    ],
})

display(exp_summary)

alpha = 0.05
decision = "Reject H0" if p_val < alpha else "Fail to reject H0"
print(
    f"Interpretation ({decision} at alpha={alpha}): Group B converts at {p_b*100:.2f}% vs. "
    f"Group A at {p_a*100:.2f}% (a relative lift of {rel_lift*100:+.2f}%, 95% CI [{ci_low*100:+.2f} pp, {ci_high*100:+.2f} pp], "
    f"chi2={chi2_stat:.4f}, p={p_val:.4f}). Groups are well balanced across counties, channels, and signup months, "
    f"supporting a rollout evaluation while monitoring post-signup transaction quality."
)

conn.close()

--- Group Balance: User Counts ---


,users
group,
B,1557
A,1443


--- Group Balance: County Share (%) ---


group,A,B
county,,
Eldoret,18.09,17.92
Kisumu,18.43,20.55
Mombasa,21.97,21.32
Nairobi,21.55,20.75
Nakuru,19.96,19.46


--- Group Balance: Acquisition Channel Share (%) ---


group,A,B
acquisition_channel,,
Agent,24.53,24.21
Referral,23.63,24.34
Search,26.33,27.87
Social,25.50,23.57


--- Group Balance: Signup Month Share (%) ---


group,A,B
signup_month,,
2026-01,26.06,25.50
2026-02,23.42,23.44
2026-03,24.39,25.24
2026-04,26.13,25.82


,Metric,Value
0,Group A Users,"1,443"
1,Group A Conversions,151
2,Group A Conversion Rate,10.46%
3,Group B Users,"1,557"
4,Group B Conversions,215
5,Group B Conversion Rate,13.81%
6,Absolute Conversion Difference (B - A),+3.34 percentage points (+0.0334)
7,Relative Lift ((B - A) / A),+31.96%
8,Chi-Square Statistic (df=1),7.8197
9,p-value,0.0052


Interpretation (Reject H0 at alpha=0.05): Group B converts at 13.81% vs. Group A at 10.46% (a relative lift of +31.96%, 95% CI [+1.01 pp, +5.67 pp], chi2=7.8197, p=0.0052). Groups are well balanced across counties, channels, and signup months, supporting a rollout evaluation while monitoring post-signup transaction quality.
